## Evaluating generative text models

### Using GPT to generate text

In [29]:
import torch.nn as nn
import torch
import tiktoken
from llmarch import GPTModel,generate_text_simple

In [22]:
# #[Text Generation->Text Evaluation->Training and Validation Losses]->LLM Training Function->Text gen Strategies->Weight saving and Loading->Pretrained weights from OpenAI
# Training and Validation Losses-Evaluate how well the model performs
# LLM Training Function-Train the model to generate human like text
# Text gen Strategies-Implement additional LLM text generation strategies to reduce training data memorization
# Weight saving and Loading-Implement functions to save and load the LLM weights to use or continue training the LLM later
# Pretrained weights from OpenAI-Load pretrained weights from OpenAI into our LLM Model

In [23]:
print("Hi")

Hi


In [24]:
from importlib.metadata import version
pkgs=[
    "matplotlib",
    "numpy",
    "tiktoken",
    "torch",
    "tensorflow"
]
for p in pkgs:
    print(f"{p} version:{version(p)}")

matplotlib version:3.10.8
numpy version:2.2.6
tiktoken version:0.12.0
torch version:2.7.1+cu118
tensorflow version:2.20.0


In [25]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [28]:
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
model.eval();

In [30]:
def text_to_token_ids(text,tokenizer):
    encoded=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
    encoded_tensor=torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

In [33]:
text="Every effort moves you"
tokenizer=tiktoken.get_encoding("gpt2")

token_ids=text_to_token_ids(text,tokenizer)
token_ids

tensor([[6109, 3626, 6100,  345]])

In [34]:
def token_ids_to_text(token_ids,tokenizer):
    flat=token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

token_ids_to_text(token_ids,tokenizer)

'Every effort moves you'

In [36]:
start_context="Hello, I am"
token_ids=generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context,tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

In [39]:
token_ids.squeeze(0).shape

torch.Size([14])

In [40]:
token_ids_to_text(token_ids,tokenizer)

'Hello, I am Laur inhab DistrinetalkQueue bear confidentlyggyenium'

### Calculate text Generation Loss(Cross entropy and Perplexity)

In [42]:
#Cross entropy 
#Now we measure the quality of each token 
inputs=torch.tensor([[16833,3626,6100],#input1-Every effort moves
                     [40,1107,588]])#input2-I really like
targets=torch.tensor([[3626,6100,345],#Target1-effort moves you
                      [1107,588,11311]])#Target2-really like chocolates


In [43]:
with torch.no_grad():
    logits=model(inputs)

In [44]:
logits.shape

torch.Size([2, 3, 50257])

In [45]:
probas=torch.softmax(logits,dim=-1)
probas.shape

torch.Size([2, 3, 50257])

In [ ]:
probas #refer to slak-own

tensor([[[1.8851e-05, 1.5173e-05, 1.1687e-05,  ..., 2.2408e-05,
          6.9776e-06, 1.8775e-05],
         [9.1574e-06, 1.0062e-05, 7.8784e-06,  ..., 2.9089e-05,
          6.0105e-06, 1.3569e-05],
         [2.9875e-05, 8.8504e-06, 1.5741e-05,  ..., 3.5458e-05,
          1.4094e-05, 1.3525e-05]],

        [[1.2561e-05, 2.0537e-05, 1.4331e-05,  ..., 1.0388e-05,
          3.4784e-05, 1.4238e-05],
         [7.2733e-06, 1.7863e-05, 1.0564e-05,  ..., 2.1206e-05,
          1.1390e-05, 1.5558e-05],
         [2.9495e-05, 3.3605e-05, 4.1032e-05,  ..., 6.5251e-06,
          5.8202e-05, 1.3697e-05]]])

In [47]:
token_ids=torch.argmax(probas,dim=-1,keepdim=True)
print("Token IDs:\n",token_ids)


Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])
